In [ ]:
import argparse
import os
import sys
import cv2
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
import torch.nn as nn
from tqdm import tqdm
from utils.dataloader_cls import BUS_CEUS_Classification, BUS_CEUS_Classification_Images
from models.tsnet import tsnet_rx18, tsnet_rx50
from models.asycmst import AsyCMST
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score, roc_auc_score
from utils.eval_metrics import confusion_matrix, draw_roc_curve
import datetime
from torch.utils.tensorboard import SummaryWriter
import yaml
from easydict import EasyDict as edict

In [ ]:
config_path = './config/test_asycmst_bus_ceus.yaml'
config = edict(yaml.load(open(config_path), Loader=yaml.FullLoader))

model_dict = {
              'tsnet_rx18': tsnet_rx18(num_classes=2, sample_size=224, sample_duration=16),
              'tsnet_rx50': tsnet_rx50(num_classes=2, sample_size=224, sample_duration=16),
              'asycmst_r18_384_6_6': AsyCMST(num_classes=2, sample_size=224, sample_duration=16, embed_dim=384, num_layers=6, num_heads=6)
              }

task = config.task
model_name = config.model.name
device = torch.device(config.device.single if torch.cuda.is_available() else "cpu")

data_root_dir = config.data.root_dir
test_label = config.data.test_label

n_classes = config.data.num_class
batch_size = config.params.batch_size
num_epoches = config.params.epoch
learning_rate = config.params.lr

vid_size = config.data.size
num_frm = config.data.num_frm

net = model_dict[model_name].to(device)
ckpt_path = config.model.load_path
if ckpt_path is not None:
    print(f"Loading model from {ckpt_path}...")
    ckpt = torch.load(ckpt_path, map_location=device)
    net.load_state_dict(ckpt)
    
optimizer = torch.optim.RMSprop(net.parameters(), lr=learning_rate, momentum=0.9, weight_decay=1e-4)
# optimizer = torch.optim.Adam(net.parameters(), lr=learning_rate, weight_decay=1e-4, betas=(0.9, 0.999), eps=1e-08)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', factor=0.5, patience=3)
loss_weight = torch.tensor([1.0, 1.0])
criterion = nn.CrossEntropyLoss(loss_weight).to(device)
timestamp = datetime.datetime.now().strftime('%Y_%m_%d_%H_%M_%S')
save_name = f'{timestamp}_{model_name}'
writer_dir = os.path.join('runs', task, save_name)
# writer = SummaryWriter(log_dir=writer_dir)
save_dir = os.path.join(writer_dir, 'ckpt')
log_dir = os.path.join(writer_dir, 'logs')

try:
    os.makedirs(save_dir)
except:
    pass
try:
    os.makedirs(log_dir)
except:
    pass

test_dataset = BUS_CEUS_Classification_Images(
    root_dir=data_root_dir, 
    label_csv=test_label,
    video_size=vid_size,
    num_frm=num_frm,
    mode='eval',
    augment=config.data.augment)

test_num = len(test_dataset)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size, 
    shuffle=False,
    num_workers=8)

print(f"Using {test_num} samples for testing, {n_classes} classes.")\

In [ ]:
# Test
net.eval()
test_loss = 0.0
with tqdm(total=len(test_dataset), desc=f'Validation', unit='imgs') as pbar_valid:
    valid_loss = 0.0
    pred_list = []
    pred_score_list = []
    label_list = []
    for batch in test_loader:
        images_ceus, images_bus, labels = batch['segment_ceus'], batch['segment_bus'], batch['label']
        images_ceus, images_bus = images_ceus.to(device), images_bus.to(device)
        labels = labels.to(device)
        with torch.no_grad():
            logits = net(images_bus, images_ceus)
            loss = criterion(logits, labels)
        test_loss += loss.detach().item()
        pred_score = torch.softmax(logits, dim=1)
        pred_results = torch.argmax(pred_score, dim=1)
        pred_list.extend(pred_results.tolist())
        pred_score_list.extend(pred_score[:,1].tolist())
        label_list.extend(labels.tolist())
        pbar_valid.update(images_bus.shape[0])

test_loss = test_loss / len(test_dataset)
test_acc = accuracy_score(label_list, pred_list)
test_rec = recall_score(label_list, pred_list)
test_pre = precision_score(label_list, pred_list, zero_division=0)
test_f1 = f1_score(label_list, pred_list)
test_auc = roc_auc_score(label_list, pred_score_list)
draw_roc_curve(label_list, pred_score_list, (2.5,2.5), os.path.join(log_dir, f'Test_Acc_{test_acc:.4f}_RoC.jpg'), 'Test')
test_cm = confusion_matrix(label_list, pred_list, [str(i) for i in range(n_classes)], (2.5,2), os.path.join(log_dir, f'Test_Acc_{test_acc:.4f}_CM.jpg'))
test_df = pd.DataFrame({'score': pred_score_list, 'pred': pred_list, 'label': label_list})
test_df.to_csv(os.path.join(log_dir, f'Test_Acc_{test_acc:.4f}.csv'), index=False)

print(f'''Test: loss:{test_loss:.8f}, Acc: {test_acc:.4f}, Pre: {test_pre:.4f}, Rec: {test_rec:.4f}, F1: {test_f1:.4f}, AUC: {test_auc:.4f}''')

# writer.close()